# Mystery & Sci-Fi — One-Click Colab Run

This is the straight-through runner for the first six-scene test. It does the setup, syncs the latest production code, checks your Colab Secret, prepares the episode, writes narration and timed captions, and creates the Google Flow manifest in one cell.

**Use:** Open in Google Colab and run the single code cell once. No separate setup cell, pasted patch, or Prepare button.

- Uses your existing Google Drive workspace and resumes files from the previous attempt.
- Uses the six-scene premise already selected.
- Keeps Gemini image API generation **off**; this does not spend separate image-generation API credits.
- Saves a persistent log in the project folder, including the last output if a run fails.
- Google Flow video generation is still a manual step after preparation; this notebook will not spend Flow credits automatically.


In [ ]:
import os, sys, json, time, shutil, subprocess, traceback, html as html_lib
from pathlib import Path
from google.colab import drive, userdata
from IPython.display import display, Markdown, HTML

# Fixed for the first six-scene test; edit these values only when making a different episode.
TOPIC = "An abandoned coastal radio station receives a warning about an event that has not happened yet."
SCENE_COUNT = 6
VOICE = "en-US-ChristopherNeural"
WHISPER_MODEL = "base.en"
WHISPER_DEVICE = "auto"
WHISPER_COMPUTE_TYPE = "int8"
WHISPER_DEVICE_PREFERENCE = "auto"
PROJECT_ID = "20261120_123020_mystery_scifi_An_abandoned_coastal_radio_station_recei"
RESUME_PROJECT_ID = PROJECT_ID
REPO_URL = "https://github.com/topboyasian-stack/gta6-autoclipper.git"
ROOT = Path("/content/gta6-autoclipper")
LOG_PATH = None

try:
    # Validate the key before Drive mounting, cloning, package installation, or API calls.
    try:
        api_key = userdata.get("GEMINI_API_KEY")
    except Exception:
        api_key = None
    api_key = str(api_key or os.environ.get("GEMINI_API_KEY", "")).strip()
    if len(api_key) < 10:
        raise RuntimeError(
            "GEMINI_API_KEY is missing or inaccessible. Add it under Colab Secrets "
            "with notebook access enabled, or provide it in the environment. "
            "The run was stopped before production work or API calls."
        )

    if not Path("/content/drive/MyDrive").is_dir():
        print("Connecting Google Drive…", flush=True)
        drive.mount("/content/drive")

    WORKSPACE = Path("/content/drive/MyDrive/GTA6_AutoClipper")
    PROJECT_DIR = WORKSPACE / "projects" / RESUME_PROJECT_ID
    OUTPUT_DIR = PROJECT_DIR / "mystery_scifi"
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    LOG_PATH = OUTPUT_DIR / "mystery_run.log"

    def note(message):
        stamp = time.strftime("%Y-%m-%d %H:%M:%S")
        line = f"[{stamp}] {message}"
        print(line, flush=True)
        with LOG_PATH.open("a", encoding="utf-8") as log:
            log.write(line + "\n")
            log.flush()

    with LOG_PATH.open("a", encoding="utf-8") as log:
        log.write("\n\n===== One-click six-scene run: " + time.strftime("%Y-%m-%d %H:%M:%S") + " =====\n")

    note("Persistent log: " + str(LOG_PATH))
    note("Project output: " + str(OUTPUT_DIR))

    def run_logged(label, command, env=None):
        note("START: " + label)
        with LOG_PATH.open("a", encoding="utf-8") as log:
            log.write("COMMAND: " + " ".join(str(part) for part in command) + "\n")
            log.flush()
            process = subprocess.Popen(
                [str(part) for part in command],
                cwd=str(ROOT) if ROOT.is_dir() else None,
                env=env,
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=True,
                bufsize=1,
                errors="replace",
            )
            log.write("PROCESS_ID: " + str(process.pid) + "\n")
            log.flush()
            print("Process started: " + str(process.pid), flush=True)
            for line in process.stdout:
                log.write(line)
                log.flush()
                print(line, end="", flush=True)
            return_code = process.wait()
            log.write("EXIT_CODE: " + str(return_code) + "\n")
            log.flush()
        if return_code != 0:
            raise RuntimeError(f"{label} failed with exit code {return_code}. See {LOG_PATH}")
        note("DONE: " + label)

    def topic_tokens(value):
        import re
        ignored = {
            "a", "an", "and", "are", "as", "at", "by", "for", "from", "in",
            "into", "is", "it", "of", "on", "or", "the", "that", "this", "to",
            "was", "were", "with",
        }
        return {
            token for token in re.findall(r"[a-z0-9]+", str(value or "").casefold())
            if len(token) > 1 and token not in ignored
        }

    def topics_differ_significantly(saved_topic, requested_topic):
        import re
        saved_normalized = " ".join(re.findall(r"[a-z0-9]+", str(saved_topic or "").casefold()))
        requested_normalized = " ".join(re.findall(r"[a-z0-9]+", str(requested_topic or "").casefold()))
        if not saved_normalized or not requested_normalized or saved_normalized == requested_normalized:
            return False
        saved_tokens = topic_tokens(saved_topic)
        requested_tokens = topic_tokens(requested_topic)
        if not saved_tokens or not requested_tokens:
            return True
        similarity = len(saved_tokens & requested_tokens) / max(1, len(saved_tokens | requested_tokens))
        return similarity < 0.30

    if (ROOT / ".git").is_dir():
        run_logged("Sync latest GitHub production code", ["git", "-C", str(ROOT), "fetch", "--depth=1", "origin", "main"])
        run_logged("Select the latest tested main commit", ["git", "-C", str(ROOT), "reset", "--hard", "FETCH_HEAD"])
    else:
        if ROOT.exists():
            shutil.rmtree(ROOT)
        run_logged("Download production code", ["git", "clone", "--depth", "1", REPO_URL, str(ROOT)])

    # Resume takes priority over the newly entered TOPIC when the saved project is
    # materially different. Warn first and preserve that plan; never delete it by count.
    plan_path = OUTPUT_DIR / "episode.json"
    saved_plan_topic = None
    resume_topic_override = False
    if plan_path.is_file():
        try:
            previous = json.loads(plan_path.read_text(encoding="utf-8"))
            saved_plan_topic = str(previous.get("topic", "")).strip() or None
            previous_count = len(previous.get("scenes", []))
        except Exception as parse_error:
            previous_count = -1
            note(f"WARNING: Existing episode.json could not be read ({type(parse_error).__name__}: {parse_error}); it will be backed up before a new plan is generated.")
        if saved_plan_topic and topics_differ_significantly(saved_plan_topic, TOPIC):
            resume_topic_override = True
            note(
                f"WARNING: RESUME_PROJECT_ID='{RESUME_PROJECT_ID}' is overriding TOPIC. "
                f"Saved project plan topic: {saved_plan_topic!r}. Current TOPIC: {TOPIC!r}. "
                "The existing project's plan will be reused unchanged."
            )
        elif previous_count != SCENE_COUNT:
            backup = OUTPUT_DIR / ("episode.backup.before_scene_count_change_" + time.strftime("%Y%m%d_%H%M%S") + ".json")
            shutil.copy2(plan_path, backup)
            plan_path.unlink()
            saved_plan_topic = None
            note(f"Existing plan had {previous_count} scenes; backed it up to {backup} before generating the requested {SCENE_COUNT}-scene plan.")
        else:
            note("Found a saved plan with the requested topic and scene count; it will be reused.")

    PROJECT_DIR.mkdir(parents=True, exist_ok=True)
    (PROJECT_DIR / "project_state.json").write_text(
        json.dumps({
            "project_name": RESUME_PROJECT_ID,
            "mode": "Mystery & Sci-Fi Automation",
            "story_title": (saved_plan_topic or TOPIC)[:160],
            "mystery_output": str(OUTPUT_DIR),
            "updated_at": time.strftime("%Y-%m-%d %H:%M:%S"),
            "setup": "one-click-colab",
        }, indent=2),
        encoding="utf-8",
    )

    if not shutil.which("ffmpeg") or not shutil.which("ffprobe"):
        run_logged(
            "Install FFmpeg",
            ["bash", "-lc", "apt-get update -qq && apt-get install -y -qq ffmpeg fonts-dejavu-core"],
        )
    else:
        note("FFmpeg and ffprobe are available.")

    requirements_path = ROOT / "requirements.txt"
    if not requirements_path.is_file():
        raise FileNotFoundError(
            f"Repository requirements file is missing: {requirements_path}. "
            "The notebook will not fall back to unpinned ad-hoc pip installs."
        )
    note("Install pinned production dependencies from the repository requirements.txt.")
    run_logged(
        "Install pinned Python dependencies",
        [
            sys.executable, "-m", "pip", "install", "-q",
            "--disable-pip-version-check", "-r", str(requirements_path),
        ],
    )

    # Probe host GPU visibility, then verify that the installed CTranslate2 build
    # can actually use CUDA. A detected host GPU alone is not enough.
    gpu_signal = False
    gpu_signal_source = []
    try:
        nvidia_probe = subprocess.run(
            ["nvidia-smi", "-L"], capture_output=True, text=True, timeout=10, check=False
        )
        if nvidia_probe.returncode == 0 and nvidia_probe.stdout.strip():
            gpu_signal = True
            gpu_signal_source.append("nvidia-smi")
    except (FileNotFoundError, subprocess.SubprocessError):
        pass
    try:
        import torch
        if torch.cuda.is_available():
            gpu_signal = True
            gpu_signal_source.append("torch.cuda.is_available")
    except Exception:
        pass
    ctranslate_cuda_count = 0
    try:
        import ctranslate2
        ctranslate_cuda_count = int(ctranslate2.get_cuda_device_count())
        if ctranslate_cuda_count > 0:
            gpu_signal = True
            gpu_signal_source.append("CTranslate2")
    except Exception as gpu_error:
        note(f"WARNING: CTranslate2 CUDA probe failed; CPU fallback will be used. {type(gpu_error).__name__}: {gpu_error}")

    if gpu_signal and ctranslate_cuda_count > 0:
        WHISPER_DEVICE = "cuda"
        WHISPER_COMPUTE_TYPE = "float16"
        note(
            f"Whisper GPU detected ({', '.join(gpu_signal_source)}; "
            f"{ctranslate_cuda_count} CTranslate2 CUDA device(s)); "
            "preferred device=cuda, compute_type=float16."
        )
    else:
        WHISPER_DEVICE = "cpu"
        WHISPER_COMPUTE_TYPE = "int8"
        if gpu_signal and ctranslate_cuda_count < 1:
            note("WARNING: A GPU signal exists, but CTranslate2 cannot use CUDA; falling back to CPU int8.")
        else:
            note("No usable CUDA GPU detected; Whisper will use CPU int8.")
    note("Whisper device preference is 'auto'; a CUDA initialization failure will retry on CPU int8.")

    # Keep the legacy project variable synchronized with the explicit resume selector.
    PROJECT_ID = RESUME_PROJECT_ID
    env = os.environ.copy()
    env.update({
        "PYTHONUNBUFFERED": "1",
        "GEMINI_API_KEY": api_key,
        "ALLOW_PAID_IMAGE_API": "false",
        "IMAGE_PROVIDER": "manual_flow",
        "VIDEO_PROVIDER": "manual_flow",
        "ENABLE_PAID_VIDEO_API": "0",
        "WHISPER_DEVICE": WHISPER_DEVICE,
        "WHISPER_COMPUTE_TYPE": WHISPER_COMPUTE_TYPE,
        "MYSTERY_WHISPER_DEVICE": WHISPER_DEVICE_PREFERENCE,
        "MYSTERY_WHISPER_MODEL": WHISPER_MODEL,
        "RESUME_PROJECT_ID": PROJECT_ID,
        "MYSTERY_NARRATOR_VOICE": VOICE,
        "MYSTERY_MUSIC_TRACK": str(OUTPUT_DIR / "no_music_selected.mp3"),
    })
    command = [
        sys.executable, str(ROOT / "main.py"),
        "--stage", "prepare",
        "--topic", TOPIC,
        "--scene-count", str(SCENE_COUNT),
        "--output", str(OUTPUT_DIR),
        "--voice", VOICE,
        "--whisper-model", WHISPER_MODEL,
        "--whisper-device", WHISPER_DEVICE_PREFERENCE,
        "--max-scenes-per-flow-run", "10",
    ]
    # Deliberately omit --allow-paid-images.
    run_logged("Prepare six-scene episode, narration, captions, and Flow manifest", command, env=env)

    episode_path = OUTPUT_DIR / "episode.json"
    if not episode_path.is_file() or episode_path.stat().st_size == 0:
        raise RuntimeError(f"Required episode plan is missing or empty: {episode_path}")
    episode = json.loads(episode_path.read_text(encoding="utf-8"))
    actual_scene_count = len(episode.get("scenes", []))
    if actual_scene_count < 1:
        raise RuntimeError("episode.json contains no scenes.")
    if not resume_topic_override and actual_scene_count != SCENE_COUNT:
        raise RuntimeError(f"Expected {SCENE_COUNT} scenes, found {actual_scene_count}.")

    # Do not declare SUCCESS until each required deliverable exists and is non-empty.
    required_files = [
        OUTPUT_DIR / "episode.json",
        OUTPUT_DIR / "episode_brief.md",
        OUTPUT_DIR / "assets_generation_plan.md",
        OUTPUT_DIR / "audio" / "narration_manifest.json",
        OUTPUT_DIR / "flow_upload_manifest.json",
        OUTPUT_DIR / "flow_upload_manifest.txt",
        OUTPUT_DIR / "flow_master_prompt.md",
    ]
    missing_or_empty = [
        str(path) for path in required_files
        if not path.is_file() or path.stat().st_size == 0
    ]
    if missing_or_empty:
        raise RuntimeError(
            "Preparation finished but required artifacts are missing or empty: "
            + ", ".join(missing_or_empty)
        )

    audio_dir = OUTPUT_DIR / "audio"
    missing_audio = []
    for scene_number in range(1, actual_scene_count + 1):
        scene_id = f"scene_{scene_number:03d}"
        for suffix in (".mp3", ".wav"):
            audio_path = audio_dir / f"{scene_id}{suffix}"
            if not audio_path.is_file() or audio_path.stat().st_size == 0:
                missing_audio.append(str(audio_path))
    if missing_audio:
        raise RuntimeError(
            "Narration audio is incomplete; these expected per-scene files are missing or empty: "
            + ", ".join(missing_audio)
        )

    captions_srt = OUTPUT_DIR / "captions" / "subtitles.srt"
    captions_ass = OUTPUT_DIR / "captions" / "subtitles.ass"
    if not any(path.is_file() and path.stat().st_size > 0 for path in (captions_srt, captions_ass)):
        raise RuntimeError("Caption generation did not produce a non-empty subtitles.srt or subtitles.ass file.")

    # Record the plan actually used, not the TOPIC value when a resume override occurred.
    state_topic = str(episode.get("topic") or saved_plan_topic or TOPIC)
    PROJECT_DIR.mkdir(parents=True, exist_ok=True)
    (PROJECT_DIR / "project_state.json").write_text(
        json.dumps({
            "project_name": RESUME_PROJECT_ID,
            "mode": "Mystery & Sci-Fi Automation",
            "story_title": state_topic[:160],
            "mystery_output": str(OUTPUT_DIR),
            "updated_at": time.strftime("%Y-%m-%d %H:%M:%S"),
            "setup": "one-click-colab",
            "resume_topic_override": resume_topic_override,
            "whisper_device": WHISPER_DEVICE,
            "whisper_compute_type": WHISPER_COMPUTE_TYPE,
            "paid_image_api": False,
            "image_provider": "manual_flow",
            "video_provider": "manual_flow",
            "paid_video_api": False,
        }, indent=2),
        encoding="utf-8",
    )

    note("SUCCESS: every required episode, narration, caption, and Google Flow artifact exists and is non-empty.")
    display(Markdown(
        "## Episode preparation complete\n\n"
        f"**Episode:** {episode.get('title', 'Untitled')}\n\n"
        f"**Plan topic actually used:** {state_topic}\n\n"
        f"**Scenes:** {actual_scene_count}\n\n"
        f"**Resume project:** {RESUME_PROJECT_ID}\n\n"
        f"**Project folder:** {OUTPUT_DIR}\n\n"
        f"**Flow manifest:** {OUTPUT_DIR / 'flow_upload_manifest.txt'}\n\n"
        f"**Master prompt:** {OUTPUT_DIR / 'flow_master_prompt.md'}\n\n"
        f"**Narration audio:** {audio_dir} (per-scene MP3 and WAV files)\n\n"
        f"**Captions:** {captions_srt if captions_srt.is_file() else captions_ass}\n\n"
        f"**Whisper:** {WHISPER_DEVICE} / {WHISPER_COMPUTE_TYPE} (automatic CPU fallback enabled)\n\n"
        f"**Persistent run log:** {LOG_PATH}\n\n"
        "Gemini image API calls remain disabled. Google Flow video generation remains manual."
    ))

    def display_copy_panel(title, file_path, panel_id):
        content = file_path.read_text(encoding="utf-8")
        escaped_content = html_lib.escape(content, quote=True)
        rows = min(32, max(10, len(content.splitlines()) + 1))
        panel_html = f"""
        <details style="margin:12px 0;padding:12px;border:1px solid #999;border-radius:8px;">
          <summary style="cursor:pointer;font-weight:600;">{html_lib.escape(title)}</summary>
          <p style="margin:10px 0 6px;">Copy the full file without leaving Colab.</p>
          <textarea id="{panel_id}" readonly spellcheck="false" rows="{rows}"
            style="box-sizing:border-box;width:100%;font-family:monospace;font-size:12px;white-space:pre;overflow:auto;">{escaped_content}</textarea>
          <button type="button"
            onclick="const box=document.getElementById('{panel_id}'); box.focus(); box.select(); box.setSelectionRange(0, box.value.length); navigator.clipboard.writeText(box.value).then(()=>{{this.textContent='Copied';}}).catch(()=>{{this.textContent='Select text and press Ctrl+C';}});"
            style="margin-top:8px;padding:7px 12px;cursor:pointer;">Copy {html_lib.escape(title)}</button>
        </details>
        """
        display(HTML(panel_html))

    display_copy_panel("Google Flow master prompt", OUTPUT_DIR / "flow_master_prompt.md", "flow-master-prompt-copy")
    display_copy_panel("Google Flow upload manifest", OUTPUT_DIR / "flow_upload_manifest.txt", "flow-upload-manifest-copy")

    print("\nVerified artifact checklist:")
    for path in required_files:
        print(f"  [OK] {path} ({path.stat().st_size:,} bytes)")
    print(f"  [OK] {actual_scene_count} MP3 narration files and {actual_scene_count} WAV audio files")
    print(f"  [OK] Caption file: {captions_srt if captions_srt.is_file() else captions_ass}")
    print("  [OK] Flow prompt and manifest are available above in copy-friendly panels.")
    print("Google Drive project folder: " + str(PROJECT_DIR))
    print("Episode output folder: " + str(OUTPUT_DIR))
    print("Persistent run log: " + str(LOG_PATH))

except Exception as exc:
    error = f"FAILED: {type(exc).__name__}: {exc}"
    print("\n" + error, flush=True)
    try:
        with LOG_PATH.open("a", encoding="utf-8") as log:
            log.write("\n" + error + "\n")
            log.write(traceback.format_exc() + "\n")
            log.flush()
        tail = LOG_PATH.read_text(encoding="utf-8", errors="replace").splitlines()[-35:]
        print("\nLast log lines:", flush=True)
        print("\n".join(tail), flush=True)
        print("\nFull persistent log: " + str(LOG_PATH), flush=True)
    except Exception:
        print(traceback.format_exc(), flush=True)
